# Lesson 7a: Sequence Models — Theory

Every network built in 0a–6b takes a fixed-size input and produces an
output in one shot: an image in, a class score out. Language, audio and
time series are different in kind — a sentence has no fixed length, and
the *order* of its elements carries meaning a fixed-size input vector
cannot represent. Padding "the cat sat" and "the cat sat on the mat and
watched the birds" to a common length does not fix this: a dense layer
applied to input slot 3 shares nothing with the same layer applied to
slot 4, so every position must relearn the same pattern from scratch. A
**recurrent** network instead applies the *same* small set of weights at
every time step, carrying a **hidden state** forward as a compressed
summary of everything seen so far — the same weight-sharing idea that
made convolution (5a) reuse one kernel across every spatial position, now
reused across every point in time. This lesson derives that recurrence,
follows its gradient backward through time to find out why it struggles
on long sequences, and derives the gated cells — LSTM and GRU — built
specifically to fix it.

By the end of this notebook you will have:
- derived the **RNN recurrence** and verified a from-scratch forward step
  against `torch.nn.RNNCell`,
- derived **backpropagation through time** and the **Jacobian-product**
  argument for why its gradients vanish or explode over long sequences,
- derived the **LSTM gates** and the **GRU** simplification, and shown why
  the LSTM's additive cell-state path preserves gradient magnitude that a
  plain RNN's multiplicative path does not, and
- implemented a **character-level RNN's forward and backward pass from
  scratch in NumPy**, verified against PyTorch autograd, then trained and
  sampled from it.

## Introduction

Weight sharing across time is the same idea convolution used for weight
sharing across space: rather than a separate parameter for every
position, one small set of parameters is reused everywhere, so whatever
the network learns at one point is available everywhere else for free.
A recurrent network's "everywhere" is every time step — the same three
weight matrices process step 1, step 2 and step 1,000, differing only in
what hidden state they receive as input. That hidden state is the
mechanism meant to carry information from arbitrarily far in the past
forward to the present step — in principle. Whether it actually does so
over long sequences is the question this lesson answers: first with a
Jacobian-product argument for why a plain RNN usually does not, and then
with the gated architectures built specifically around that failure.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init,
# training, sampling) is reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

### A tiny text corpus

No download is needed: a single public-domain paragraph (the opening of
*Pride and Prejudice*) is embedded directly, giving a small, fixed,
reproducible character vocabulary to derive and train on. It is short by
design — this lesson is about the mechanics of recurrence, not about
training a good language model (7b does that, at word-model scale, with
PyTorch).

In [ ]:
CORPUS = (
    'It is a truth universally acknowledged, that a single man in possession ' +
    'of a good fortune, must be in want of a wife. However little known the ' +
    'feelings or views of such a man may be on his first entering a ' +
    'neighbourhood, this truth is so well fixed in the minds of the ' +
    'surrounding families, that he is considered as the rightful property of ' +
    'some one or other of their daughters. "My dear Mr. Bennet," said his ' +
    'lady to him one day, "have you heard that Netherfield Park is let at ' +
    'last?" Mr. Bennet replied that he had not. "But it is," returned she; ' +
    '"for Mrs. Long has just been here, and she told me all about it." Mr. ' +
    'Bennet made no answer. "Do not you want to know who has taken it?" cried ' +
    'his wife impatiently. "You want to tell me, and I have no objection to ' +
    'hearing it."'
)

chars = sorted(set(CORPUS))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return np.array([stoi[c] for c in s], dtype=np.int64)

def decode(idxs):
    return "".join(itos[int(i)] for i in idxs)

data = encode(CORPUS)
print(f"corpus length: {len(CORPUS)} characters, vocabulary: {vocab_size} unique characters")
print(repr(CORPUS[:60]))

## The Recurrent Cell

A recurrent cell maps an input $x_t \in \mathbb{R}^{d}$ and the previous
hidden state $h_{t-1} \in \mathbb{R}^{n}$ to a new hidden state and, at
every step, an output:

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h), \qquad y_t = W_{hy} h_t + b_y.$$

$W_{xh} \in \mathbb{R}^{n\times d}$, $W_{hh} \in \mathbb{R}^{n\times n}$
and $W_{hy}$, together with the biases, are the **entire** set of learned
parameters — the same four matrices are reused at every $t$, whether the
sequence has 5 steps or 5,000. $h_0$ is typically initialised to zero.
"Unrolling" the recurrence over $T$ steps just means applying this same
cell $T$ times, each time feeding forward the previous step's hidden
state — conceptually a $T$-layer network where every layer shares
identical weights, which is exactly why the gradient with respect to
$W_{hh}$ (derived next) sums a contribution from every time step, the same
way a shared convolutional kernel's gradient sums over every spatial
position it was applied to.

In [ ]:
def rnn_cell_forward(x_t, h_prev, Wxh, Whh, bh):
    return np.tanh(Wxh @ x_t + Whh @ h_prev + bh)


# Verify the from-scratch cell against torch.nn.RNNCell: PyTorch's cell
# computes tanh(W_ih x + b_ih + W_hh h + b_hh), i.e. the same recurrence
# with the bias split into two additive halves.
rng = np.random.default_rng(SEED)
input_size, hidden_size = 7, 5
torch_cell = torch.nn.RNNCell(input_size, hidden_size)

Wxh = torch_cell.weight_ih.detach().numpy()
Whh = torch_cell.weight_hh.detach().numpy()
bh = (torch_cell.bias_ih + torch_cell.bias_hh).detach().numpy()

x_np = rng.normal(size=(input_size,)).astype(np.float32)
h_np = rng.normal(size=(hidden_size,)).astype(np.float32)

h_next_scratch = rnn_cell_forward(x_np, h_np, Wxh, Whh, bh)
h_next_torch = torch_cell(torch.tensor(x_np).unsqueeze(0), torch.tensor(h_np).unsqueeze(0))

max_abs_diff = np.abs(h_next_scratch - h_next_torch.detach().numpy().squeeze(0)).max()
print(f"max abs diff vs torch.nn.RNNCell: {max_abs_diff:.2e}")
assert max_abs_diff < 1e-5

The from-scratch cell matches `torch.nn.RNNCell` to floating-point
precision — it is a literal realisation of the recurrence above, not an
approximation of it. Everything that follows (backpropagation through
time, the vanishing-gradient argument, and the character RNN) builds on
exactly this one cell, applied repeatedly.

## Backpropagation Through Time

Training a recurrent network requires the gradient of a total loss
$L = \sum_{t=1}^{T} L_t$ (one term per time step, e.g. the cross-entropy
of $y_t$ against a target) with respect to every weight matrix. Because
$h_t$ feeds forward into every future time step, $h_t$ influences the
loss both directly (through $y_t$) and indirectly (through $h_{t+1},
h_{t+2}, \dots$). The gradient with respect to a hidden state is
therefore recursive, computed backward from the last time step:

$$\frac{\partial L}{\partial h_t} = \underbrace{\frac{\partial L_t}{\partial h_t}}_{\text{direct}} + \underbrace{\left(\frac{\partial h_{t+1}}{\partial h_t}\right)^{\!\top} \frac{\partial L}{\partial h_{t+1}}}_{\text{from the future}}, \qquad \frac{\partial h_{t+1}}{\partial h_t} = \operatorname{diag}\!\left(1 - h_{t+1}^2\right) W_{hh}.$$

Running this recursion from $t=T$ down to $t=1$ — the forward recurrence
run in reverse, accumulating gradients instead of hidden states — **is**
backpropagation through time; it is not a different algorithm from
ordinary backprop, only the same chain rule applied to a computation
graph unrolled over time. Because $W_{hh}$ (and $W_{xh}$, $b_h$) are the
*same* matrix at every step, the total gradient with respect to $W_{hh}$
sums the local gradient contributed at every time step:

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \operatorname{diag}\!\left(1 - h_t^2\right) \frac{\partial L}{\partial h_t}\, h_{t-1}^{\top}.$$

In practice, sequences longer than a chosen window are trained with
*truncated* BPTT: the forward pass carries the hidden state across the
whole sequence, but gradients are only backpropagated a fixed number of
steps, trading exactness for a bounded and predictable amount of
computation per update — the training loop built later in this notebook
does exactly this.

## Vanishing Gradients over Time

Repeatedly applying the recursion above from step $T$ back to step $0$
means the gradient of the loss with respect to the *earliest* hidden
state is a **product** of $T$ Jacobians:

$$\frac{\partial h_T}{\partial h_0} = \prod_{t=1}^{T} \operatorname{diag}\!\left(1 - h_t^2\right) W_{hh}.$$

Two facts about this product decide whether a gradient signal from step
$T$ can reach step $0$ at all. First, $|1-h_t^2| \le 1$ for every $\tanh$
output, so each factor can only shrink a vector, never grow it, purely
from the nonlinearity. Second, and decisively, if the largest singular
value of $W_{hh}$ is less than 1, every factor in the product is a
contraction, and a product of $T$ contractions shrinks **geometrically**
in $T$ — after enough steps the gradient reaching early time steps is
indistinguishable from zero, and the network simply cannot learn any
dependency spanning that many steps. If $W_{hh}$'s largest singular value
exceeds 1 by enough to outweigh the $\tanh'$ shrinkage, the same product
instead grows geometrically — the exploding-gradient case, in practice
controlled with gradient clipping (used in the training loop below).

In [ ]:
def spectral_scale(n, scale, rng):
    """A random n x n matrix rescaled to have largest singular value `scale`."""
    M = rng.normal(size=(n, n))
    s_max = np.linalg.svd(M, compute_uv=False)[0]
    return M * (scale / s_max)


def jacobian_product_norms(Whh, x_seq, h0, Wxh, bh):
    """||dh_t/dh_0|| for t = 1..T, via the running Jacobian product."""
    T = len(x_seq)
    h = h0
    J = np.eye(len(h0))
    norms = []
    for t in range(T):
        h = rnn_cell_forward(x_seq[t], h, Wxh, Whh, bh)
        step_jacobian = np.diag(1 - h ** 2) @ Whh
        J = step_jacobian @ J
        norms.append(np.linalg.norm(J, ord=2))
    return np.array(norms)


rng2 = np.random.default_rng(SEED)
n, T = 20, 60
Wxh_toy = rng2.normal(size=(n, n)) * 0.1
bh_toy = np.zeros(n)
h0_toy = np.zeros(n)
x_seq_toy = [rng2.normal(size=n) * 0.1 for _ in range(T)]

plt.figure()
for scale in [0.5, 1.0, 1.5]:
    Whh_toy = spectral_scale(n, scale, rng2)
    norms = jacobian_product_norms(Whh_toy, x_seq_toy, h0_toy, Wxh_toy, bh_toy)
    plt.semilogy(np.arange(1, T + 1), norms, label=f"spectral norm(W_hh) = {scale}")
plt.xlabel("time steps back, $T$")
plt.ylabel(r"$\|\partial h_T / \partial h_0\|_2$ (log scale)")
plt.title("Gradient magnitude vs. sequence length")
plt.legend()
plt.tight_layout()
plt.show()

At spectral norm 0.5 the gradient reaching $h_0$ collapses by many
orders of magnitude within 60 steps — anything the network needed to
remember that far back is, for optimisation purposes, invisible. At 1.5
the gradient instead grows without bound, the exploding case. Only
right at the boundary does the product neither vanish nor explode — and
that boundary is a single scalar knob on one weight matrix, not something
a plain RNN can reliably self-tune while also fitting the data. This is
the concrete failure the gated architectures below are designed to
avoid.

## LSTM and GRU Gates

The Long Short-Term Memory (LSTM) cell keeps the same hidden state $h_t$
but adds a second recurrent quantity, the **cell state** $c_t$, updated
*additively* rather than through a repeated matrix multiplication and
squashing nonlinearity:

$$
\begin{aligned}
f_t &= \sigma(W_f [h_{t-1}, x_t] + b_f) &&\text{forget gate: what fraction of } c_{t-1} \text{ to keep}\\
i_t &= \sigma(W_i [h_{t-1}, x_t] + b_i) &&\text{input gate: how much of the new candidate to admit}\\
\tilde c_t &= \tanh(W_c [h_{t-1}, x_t] + b_c) &&\text{candidate: what the new information actually is}\\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde c_t &&\text{cell state update (additive)}\\
o_t &= \sigma(W_o [h_{t-1}, x_t] + b_o) &&\text{output gate: how much of } c_t \text{ to expose}\\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

Every gate is a sigmoid ($\sigma \in (0,1)$) applied to a learned linear
combination of the previous hidden state and the current input, so each
one independently learns *when* to let information through. The gradient
path that matters for the vanishing-gradient problem is $\partial
c_T/\partial c_0$, and because the cell-state recurrence is a **sum**, not
a repeated multiply-then-squash, that path is a product of the forget
gates alone: $\partial c_T/\partial c_0 = \prod_t f_t$ (elementwise). If
the network learns to keep $f_t$ close to 1 for a dimension it needs to
remember, that dimension's gradient passes through almost unchanged no
matter how large $T$ is — a "gradient highway" the plain RNN's
$\tanh'(\cdot) W_{hh}$ product does not have.

The Gated Recurrent Unit (GRU) simplifies this: it drops the separate
cell state and merges the forget/input gates into a single **update
gate** $z_t$, plus a **reset gate** $r_t$ that controls how much of the
past hidden state is used when forming the candidate:

$$
\begin{aligned}
z_t &= \sigma(W_z [h_{t-1}, x_t] + b_z) &&\text{update gate}\\
r_t &= \sigma(W_r [h_{t-1}, x_t] + b_r) &&\text{reset gate}\\
\tilde h_t &= \tanh(W_h [r_t \odot h_{t-1}, x_t] + b_h)\\
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde h_t
\end{aligned}
$$

$h_t$ is again an elementwise blend of the old state and a new candidate,
governed by $z_t$ — the same additive, gate-controlled gradient path as
the LSTM's cell state, with one fewer gate and no separate cell state to
maintain. In practice a GRU trains faster and uses fewer parameters per
layer than an LSTM for comparable accuracy on many tasks, which is why it
remains a common default when the LSTM's extra capacity is not needed.

In [ ]:
# A toy illustration of the two gradient paths, holding sequence length T
# fixed: the plain RNN's path multiplies tanh'(.) (<=1) by W_hh's spectral
# norm at every step; the gated path multiplies forget gates alone. Forget
# gates near 1 are exactly what makes the gated path decay far slower.
T = 60
rnn_factor = 0.9   # a representative |tanh'(.)| * spectral-norm(W_hh) per step
forget_gate = 0.98  # a representative learned forget-gate value near 1

steps = np.arange(1, T + 1)
rnn_path = rnn_factor ** steps
gated_path = forget_gate ** steps

plt.figure()
plt.semilogy(steps, rnn_path, label=f"plain RNN path (per-step factor {rnn_factor})")
plt.semilogy(steps, gated_path, label=f"LSTM cell-state path (forget gate {forget_gate})")
plt.xlabel("time steps back, $T$")
plt.ylabel("relative gradient magnitude (log scale)")
plt.title("Multiplicative vs. additive gradient path")
plt.legend()
plt.tight_layout()
plt.show()

print(f"after {T} steps: plain RNN retains {rnn_path[-1]:.2e}, gated path retains {gated_path[-1]:.2e}")

Both paths decay geometrically — an LSTM does not make long-range
dependencies free — but a forget gate the network has learned to keep
near 1 decays dramatically more slowly than a per-step factor already
below 1 from an untrainable nonlinearity bound. That gap, compounded over
tens or hundreds of steps, is the entire practical difference between a
plain RNN losing a signal within a few dozen steps and a gated cell
carrying it much further.

## A Character RNN from Scratch

What remains is to implement the plain RNN cell's forward *and* backward
pass over a full sequence in NumPy — no autograd — verify the manually
derived gradients against PyTorch's autograd on an identical computation
graph, and then actually train the result on the tiny corpus above.

In [ ]:
def one_hot(idx, size):
    v = np.zeros(size, dtype=np.float64)
    v[idx] = 1.0
    return v


def rnn_sequence_forward(input_idxs, target_idxs, h_prev, params):
    """Forward pass over a sequence, computing per-step softmax cross-entropy loss.

    Returns the total loss and a cache of everything the backward pass needs.
    """
    Wxh, Whh, Why, bh, by = params
    xs, hs, ys, ps = {}, {-1: h_prev}, {}, {}
    loss = 0.0
    for t, (ix, target) in enumerate(zip(input_idxs, target_idxs)):
        xs[t] = one_hot(ix, Wxh.shape[1])
        hs[t] = np.tanh(Wxh @ xs[t] + Whh @ hs[t - 1] + bh)
        ys[t] = Why @ hs[t] + by
        # numerically stable softmax
        shifted = ys[t] - ys[t].max()
        exp = np.exp(shifted)
        ps[t] = exp / exp.sum()
        loss += -np.log(ps[t][target] + 1e-12)
    cache = dict(xs=xs, hs=hs, ps=ps, targets=target_idxs, params=params)
    return loss, cache


def rnn_sequence_backward(cache):
    """Backpropagation through time for the sequence above."""
    xs, hs, ps, targets, (Wxh, Whh, Why, bh, by) = (
        cache["xs"], cache["hs"], cache["ps"], cache["targets"], cache["params"]
    )
    dWxh = np.zeros_like(Wxh)
    dWhh = np.zeros_like(Whh)
    dWhy = np.zeros_like(Why)
    dbh = np.zeros_like(bh)
    dby = np.zeros_like(by)
    dh_next = np.zeros(Whh.shape[0])

    T = len(targets)
    for t in reversed(range(T)):
        dy = ps[t].copy()
        dy[targets[t]] -= 1.0  # softmax + cross-entropy gradient
        dWhy += np.outer(dy, hs[t])
        dby += dy

        dh = Why.T @ dy + dh_next               # direct + from-the-future term
        dh_raw = (1 - hs[t] ** 2) * dh           # through tanh
        dbh += dh_raw
        dWxh += np.outer(dh_raw, xs[t])
        dWhh += np.outer(dh_raw, hs[t - 1])
        dh_next = Whh.T @ dh_raw

    for grad in (dWxh, dWhh, dWhy, dbh, dby):
        np.clip(grad, -5, 5, out=grad)  # gradient clipping, per the exploding-gradient argument above
    return dWxh, dWhh, dWhy, dbh, dby, hs[T - 1]

Verification follows the same pattern as 5a's convolution check: build
the identical computation with `torch` tensors and `requires_grad=True`,
call `.backward()`, and compare every gradient the manual backward pass
above computed against what autograd computed for the same graph.

In [ ]:
def torch_reference_grads(input_idxs, target_idxs, h_prev, params):
    Wxh_np, Whh_np, Why_np, bh_np, by_np = params
    Wxh = torch.tensor(Wxh_np, requires_grad=True)
    Whh = torch.tensor(Whh_np, requires_grad=True)
    Why = torch.tensor(Why_np, requires_grad=True)
    bh = torch.tensor(bh_np, requires_grad=True)
    by = torch.tensor(by_np, requires_grad=True)
    h = torch.tensor(h_prev)

    loss = torch.zeros(())
    for ix, target in zip(input_idxs, target_idxs):
        x = torch.zeros(Wxh.shape[1], dtype=torch.float64)
        x[ix] = 1.0
        h = torch.tanh(Wxh @ x + Whh @ h + bh)
        y = Why @ h + by
        loss = loss + torch.nn.functional.cross_entropy(y.unsqueeze(0), torch.tensor([target]))
    loss.backward()
    return Wxh.grad.numpy(), Whh.grad.numpy(), Why.grad.numpy(), bh.grad.numpy(), by.grad.numpy()


rng3 = np.random.default_rng(SEED)
hidden_size_check, seq_len_check = 8, 12
Wxh_c = rng3.normal(size=(hidden_size_check, vocab_size)) * 0.1
Whh_c = rng3.normal(size=(hidden_size_check, hidden_size_check)) * 0.1
Why_c = rng3.normal(size=(vocab_size, hidden_size_check)) * 0.1
bh_c = np.zeros(hidden_size_check)
by_c = np.zeros(vocab_size)
h0_c = np.zeros(hidden_size_check)
params_c = (Wxh_c, Whh_c, Why_c, bh_c, by_c)

seq_idxs = data[:seq_len_check]
target_idxs = data[1:seq_len_check + 1]

loss_scratch, cache = rnn_sequence_forward(seq_idxs, target_idxs, h0_c, params_c)
grads_scratch = rnn_sequence_backward(cache)[:5]
grads_torch = torch_reference_grads(seq_idxs, target_idxs, h0_c, params_c)

names = ["dWxh", "dWhh", "dWhy", "dbh", "dby"]
for name, g_s, g_t in zip(names, grads_scratch, grads_torch):
    max_diff = np.abs(g_s - g_t).max()
    print(f"{name}: max abs diff vs torch autograd = {max_diff:.2e}")
    assert max_diff < 1e-6, f"{name} mismatch"
print(f"\nforward loss matches too: scratch cross-entropy sum = {loss_scratch:.6f}")

Both the forward loss and every one of the five manually derived
gradients match PyTorch's autograd to floating-point precision — the
scratch implementation is a literal realisation of backpropagation
through time, not an approximation of it. It can now be trained for
real.

In [ ]:
def train_char_rnn(data, vocab_size, hidden_size=64, seq_length=25,
                    epochs=60, lr=0.1, seed=SEED):
    rng = np.random.default_rng(seed)
    Wxh = rng.normal(size=(hidden_size, vocab_size)) * 0.01
    Whh = rng.normal(size=(hidden_size, hidden_size)) * 0.01
    Why = rng.normal(size=(vocab_size, hidden_size)) * 0.01
    bh = np.zeros(hidden_size)
    by = np.zeros(vocab_size)

    loss_history = []
    n_chunks = (len(data) - 1) // seq_length
    for epoch in range(epochs):
        h_prev = np.zeros(hidden_size)
        epoch_loss = 0.0
        for chunk in range(n_chunks):
            start = chunk * seq_length
            inputs = data[start:start + seq_length]
            targets = data[start + 1:start + seq_length + 1]
            params = (Wxh, Whh, Why, bh, by)
            loss, cache = rnn_sequence_forward(inputs, targets, h_prev, params)
            dWxh, dWhh, dWhy, dbh, dby, h_prev = rnn_sequence_backward(cache)
            for param, grad in zip((Wxh, Whh, Why, bh, by), (dWxh, dWhh, dWhy, dbh, dby)):
                param -= lr * grad / seq_length
            epoch_loss += loss / seq_length
        loss_history.append(epoch_loss / n_chunks)
    return (Wxh, Whh, Why, bh, by), loss_history


trained_params, loss_history = train_char_rnn(data, vocab_size)
print(f"loss: {loss_history[0]:.3f} -> {loss_history[-1]:.3f} over {len(loss_history)} epochs")

### Training loss

In [ ]:
plt.figure()
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("mean per-character cross-entropy")
plt.title("Character RNN training loss")
plt.tight_layout()
plt.show()

Loss falls steadily as the network memorises the statistics of this one
short paragraph — with a corpus this small the goal is not a generalising
language model (7b trains one properly, at word scale, with a real
validation split) but a visible, working demonstration that the from-scratch
forward and backward pass above actually optimise something.

### Sampling from the trained network

In [ ]:
def sample(params, seed_char, n_chars, temperature=0.8, seed=SEED):
    Wxh, Whh, Why, bh, by = params
    rng = np.random.default_rng(seed)
    h = np.zeros(Whh.shape[0])
    ix = stoi[seed_char]
    out = [seed_char]
    for _ in range(n_chars):
        x = one_hot(ix, Wxh.shape[1])
        h = np.tanh(Wxh @ x + Whh @ h + bh)
        y = Why @ h + by
        logits = y / temperature
        p = np.exp(logits - logits.max())
        p /= p.sum()
        ix = rng.choice(len(p), p=p)
        out.append(itos[ix])
    return "".join(out)


print(sample(trained_params, seed_char="t", n_chars=200))

The sample is not coherent English — a 64-unit RNN trained for a few
seconds on one paragraph was never going to produce one — but it
reproduces real words, common bigrams and the corpus's punctuation
rhythm, evidence that the from-scratch gradients above genuinely moved
the parameters toward the data's structure rather than merely running
without error. 7b trains a properly regularised LSTM on more data and
reports perplexity, the standard measure of how well a language model
actually predicts held-out text.

## Key Takeaways

- **A recurrent cell shares one small set of weights across every time
  step**, the same weight-sharing principle convolution applies across
  space, and unrolling it over $T$ steps is a $T$-layer network with tied
  weights.
- **Backpropagation through time is ordinary backprop on the unrolled
  graph**: gradients w.r.t. a hidden state combine a direct term and a
  term from every future step, and the shared weight matrices' gradients
  sum a contribution from every time step.
- **Gradients vanish or explode geometrically in the number of time
  steps**, governed by the spectral norm of $W_{hh}$ — verified above by
  tracking $\|\partial h_T/\partial h_0\|$ across sequence length at three
  different weight scales.
- **LSTM and GRU gates replace the repeated multiply-and-squash update
  with an additive, gate-controlled one**: a forget gate near 1 lets
  gradient pass through a cell state almost unchanged, a "highway" a
  plain RNN's Jacobian product does not have; the GRU keeps that same
  additive path with one fewer gate and no separate cell state.
- **A from-scratch NumPy forward and backward pass through a character
  RNN matched PyTorch autograd to floating-point precision** on every one
  of five gradients, and the same implementation, trained for a few
  seconds, visibly reduced its loss and produced recognisable structure
  when sampled.